# **Credit Card Fraud Detection – ETL Pipeline**

## **Objectives**

This notebook performs the full Extract–Transform–Load (ETL) process on the raw fraud transaction datasets. The goal is to produce clean, feature-engineered datasets ready for:

* Exploratory Data Analysis (EDA)
* Hypothesis Testing
* Machine Learning
* Power BI dashboard integration

Both fraudTrain.csv and fraudTest.csv are processed using identical transformation steps to prevent data leakage and ensure modelling consistency.

## Inputs

* fraudTrain.csv
* fraudTest.csv

## Outputs

* fraud_train_processed.csv
* fraud_test_processed.csv


---

# Section 1: Data Loading and Initial Exploration (Extract)

In this section, I will load the raw credit card churn dataset and perform an initial exploration to understand the structure, data types, and basic characteristics of the data.

## Import Libraries

The following libraries are required for data manipulation and feature engineering.

In [2]:
# Import Libraries


import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

## Load Raw Data

The training and testing datasets are loaded from the raw data directory and i checked the data type and the non-null count.

In [3]:
# Load Raw Data


train_df = pd.read_csv("../data/raw/fraudTrain.csv")
test_df = pd.read_csv("../data/raw/fraudTest.csv")

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)

train_df.head()
train_df.info()

Train Shape: (1296675, 23)
Test Shape: (555719, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                 

In [4]:
# Check class imbalance to see what proportion of the dataset is associated with fraudulant transactions.

train_df["is_fraud"].value_counts(normalize=True)


is_fraud
0   0.9942
1   0.0058
Name: proportion, dtype: float64

---

### Initial Observations

- The dataset contains transactional, demographic and geographic information.
- The target variable `is_fraud` is highly imbalanced.
- Several columns contain personally identifiable information (PII) which must be removed for ethical compliance.

# Section 2: Data Cleaning & Ethical Handling (Transform – Part A)

## Ethical Data Cleaning

To ensure responsible data usage, personally identifiable information (PII) and low-signal identifiers are removed.

This reduces privacy risk and prevents the model from learning from sensitive attributes.

In [5]:
# Remove PII & Low-Signal Columns


pii_columns = [
    "first",
    "last",
    "street",
    "cc_num",
    "trans_num"
]

low_signal_columns = [
    "merchant",
    "job",
    "unix_time"
]

train_df = train_df.drop(columns=pii_columns + low_signal_columns)
test_df = test_df.drop(columns=pii_columns + low_signal_columns)

print("Remaining Columns:", train_df.shape[1])

Remaining Columns: 15


## Check for Missing Values

In [6]:
train_df.isnull().sum().sort_values(ascending=False).head(10)

Unnamed: 0               0
trans_date_trans_time    0
category                 0
amt                      0
gender                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
dtype: int64

### Observations
No significant missing values were detected.  
No imputation was required at this stage.

---

# Section 3: Feature Engineering (Transform – Part B)

## Datetime Feature Engineering

Transaction timestamps are converted to datetime format and decomposed into behavioural indicators.

In [7]:
# Datetime Processing


train_df["trans_date_trans_time"] = pd.to_datetime(train_df["trans_date_trans_time"])
test_df["trans_date_trans_time"] = pd.to_datetime(test_df["trans_date_trans_time"])

for df in [train_df, test_df]:
    df["trans_hour"] = df["trans_date_trans_time"].dt.hour
    df["trans_dayofweek"] = df["trans_date_trans_time"].dt.dayofweek
    df["trans_month"] = df["trans_date_trans_time"].dt.month

## Age Feature Engineering

Customer age is calculated using date of birth and transaction timestamp.

Raw DOB will later be removed to protect privacy.

In [8]:
# Age Calculation


for df in [train_df, test_df]:
    df["dob"] = pd.to_datetime(df["dob"])
    df["age"] = (df["trans_date_trans_time"] - df["dob"]).dt.days // 365

---

## Geographic Distance Calculation (Haversine)

Distance between customer home location and merchant location is calculated.

Unusually large distances may indicate fraudulent behaviour.

In [9]:
# Haversine Distance


def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

for df in [train_df, test_df]:
    df["home_merch_dist"] = df.apply(
        lambda row: haversine(
            row["lat"], row["long"],
            row["merch_lat"], row["merch_long"]
        ),
        axis=1
    )

---

## Log Transformation of Transaction Amount

Transaction amounts are highly skewed.

A log transformation reduces skewness and stabilises variance.

In [10]:
# Log Transform Amount


for df in [train_df, test_df]:
    df["log_amt"] = np.log1p(df["amt"])

- - -


## Final Column Cleanup

Redundant raw columns are removed after feature engineering.

In [11]:
# Drop Redundant Columns


drop_columns = [
    "trans_date_trans_time",
    "dob"
]

train_df = train_df.drop(columns=drop_columns)
test_df = test_df.drop(columns=drop_columns)

---

# Section 4: Save Processed Data (Load)

## Save Processed Datasets

The cleaned and engineered datasets are saved for:

- Exploratory Data Analysis
- Statistical testing
- Machine Learning
- Dashboard reporting

In [12]:
# Save Processed Data


train_df.to_csv("../data/processed/fraud_train_processed.csv", index=False)
test_df.to_csv("../data/processed/fraud_test_processed.csv", index=False)

print("Processed datasets saved successfully.")
print("Final Train Shape:", train_df.shape)
print("Final Test Shape:", test_df.shape)

Processed datasets saved successfully.
Final Train Shape: (1296675, 19)
Final Test Shape: (555719, 19)


# ETL Summary

This notebook completed the full ETL pipeline:

* Raw data successfully loaded  
* Personally identifiable information removed  
* Datetime features engineered  
* Customer age calculated  
* Geographic transaction distance computed  
* Transaction amounts log-transformed  
* Clean datasets saved for downstream analysis  

The processed datasets are now ready for:
* Exploratory Data Analysis (02_eda.ipynb)
* Machine Learning (03_modeling.ipynb)

All transformations were applied consistently to both training and testing datasets to prevent data leakage.